## **Harnessing Machine Learning for Seismic Phase Picking in the cloud**

Seismology is exploding with data. With thousands of sensors collecting terabytes of waveforms through networks like the SAGE facility, there’s more data than any human (or team of humans) can comb through by hand. That’s where machine learning (ML) comes in.

ML lets computers learn patterns from data, such as recognizing earthquake signals or picking out P and S wave arrivals, without being explicitly told the rules. Instead of writing rigid algorithms, we let the computer figure it out from examples.

Think of it as training a dog to sit. You don’t teach it every muscle to move; you reward it when it gets it right. Over time, it just learns. That’s exactly what we do when we train ML models on seismic data.

##### 🔹 **Train a model?**

“Training a model” means feeding it lots of labeled examples, like seismograms with known phase arrivals, and letting it learn the connection between the raw waveform and the arrival times.

At first, the model guesses randomly. But with each guess, it gets feedback (how wrong it was), and it adjusts its inner logic (called parameters) to do better next time. After enough rounds, the model gets pretty good at spotting patterns — even in brand new data it's never seen before.

##### 🔹 **Meet U-net**

When it comes to finding where something is inside messy data (like an earthquake hiding in noise), one of the most effective tools is called the U-Net. Originally created for segmenting medical images (like finding tumors in MRI scans), U-Net has found a new job in seismology: picking out seismic phases from waveform data.

How does it work? Think of it like this:

1. 👀 It zooms out to understand the big picture (what's happening over time).
2. 🔍 Then it zooms back in, focusing on fine details to pinpoint exactly where the interesting stuff is — like a P-wave arrival.

Even better, it brings along notes from the zoomed-out view so it doesn’t forget anything. This “zoom out and back in” strategy is why it's shaped like a U — wide on the sides, narrow in the middle.

U-Net works well on seismograms for a few key reasons:

- It combines context and detail — it sees the whole trace and also zooms in on precise moments.
- It’s efficient — it can learn a lot even from a relatively small number of labeled waveforms.
- It’s precise — it highlights the exact sample or time a seismic phase arrives.
- It’s flexible — it works for 1D traces, 2D waveform images, or even full event catalogs.
- All this makes it a favorite for seismic phase picking and event detection.

Training a deep learning model like U-Net takes time, power, and lots of data. Traditionally, that meant setting up local machines, transferring files, and managing storage — all of which can slow you down.

But now, SAGE data is hosted in the cloud, and that changes everything.

- You can stream waveform data directly into your ML pipeline.
- Use cloud GPUs or TPUs to train models in parallel.
- Automate your workflows using tools like Dask, Kubernetes, or prefect.io.
- Reproduce and share everything with portable, containerized environments.

In short: cloud-optimized workflows make training ML models on seismic data faster, cheaper, and more scalable than ever before.

This is how a U-net architecture works

<img src="https://oup.silverchair-cdn.com/oup/backfile/Content_public/Journal/gji/216/1/10.1093_gji_ggy423/1/m_ggy423fig5.jpeg?Expires=1754919728&Signature=41lJnaVjzCCW1i7utY-eOebfBdjeh7ykYBMBIqT0yyFLj3PCn60KlQD96BDL8kuQRxugnvm3BUxljgkI6VB6UcMNjIFXKamdpVuDpzjgwjMNwiZCdjOr1ygjxpixGkMHfzFV1w0y9JkY60gwVSZvhe4BlGfjAuuTZb9-6gVcxpLvQtFiOddFB6QpOblUD4RVFezUgbeCTa2VA4OI6O91G7DrWGR7EtSP~QAkYhqG4l~VmQTLxcZWijRHDEr~XGcPXlgR77x7NtRschgq6IHf0i7lv8F4MfbkMnvIDmkAm5m1pUQT8Q2SSyji9hT~p-NAyKhSEXuGFodua-SCpXWytg__&Key-Pair-Id=APKAIE5G5CRDK6RD3PGA" alt="DASKscaling" width="800"/>


In [2]:
!pip install torch

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

## Build the U-net

First, we will go down with the Encoder path and build the required fuction. What is the first step that you see in there? It is the convolution and Relu process but twice. We are going to call it double convolution layer. So first let's build this layer below:

In [2]:
# Import the necessary module from PyTorch
import torch.nn as nn

# Define a class to represent a double convolution block
class ConvBlock(nn.Module): # inherit the base class for neural networks for PyTorch
    """
    ┌──────────────────────────────────────────────────────────────────────────────────────────────────┐
    │ ConvBlock: The basic building brick                                                              │
    │ ------------------------------------------------------------------------------------------------ │
    │ • Two 1-D convolutions (doubleConv), each followed by ReLU.                                      │
    │ • Keeps the time length unchanged (padding=1) but learns richer representations (more channels). |                                    │
    │ • Re-using the same pattern everywhere keeps the code short and consistent.                      |                                    │
    └──────────────────────────────────────────────────────────────────────────────────────────────────┘
    """
    def __init__(self, 
                 in_channels,       # No of input channels (e.g., 1 for grayscale, 3 for RGB)
                 out_channels,      # No of output channels (i.e., number of feature maps).
                 kernel_size = 3,   # Size of the convolutional filter (default is 3x3).
                 padding = 1        # Padding added to both sides of input (default is 1 to preserve size).
                 ):
        super().__init__()  # Initialize the parent nn.Module class
        
        # ① First 1-D convolution.
        #    – Looks at a sliding 3-sample window (for kernel_size=3).
        #    – padding=1 so output length == input length.
        #    – Learns out_channel different “filters” (patterns) in parallel.
        conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)

        # ② ReLU: keeps only positive values → adds non-linearity.
        relu1 = nn.ReLU(inplace=True)

        # ③ repeat these two steps once more, so we have a 'two-layer feature extractor'
        conv2 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        relu2 = nn.ReLU(inplace=True)

        # Put the four layers into one tidy “Sequential” container.
        self.doubleConv = nn.Sequential(conv1, relu1, conv2, relu2)
        
    def forward(self, x):
        # Simply run x through the mini-network with four layers (Conv-ReLu+Conv-ReLu) defined above.
        return self.doubleConv(x) # Pass input through the doubleConv block

If you’re just getting started with CNNs or U-net, building a modular layer like this helps break down a complex architecture into simpler, reusable parts. Once you understand how a single block works, it's easier to scale this up to the full model with encoder and decoder layers.

In [3]:
# now, we are going to create the entire U-net structure

class UNet1D(nn.Module): # inherit the base class for neural networks for PyTorch
    """
    ┌────────────────────────────────────────────────────────────────────────┐
    │ 1-D U-Net (originally designed for images, adapted to time series)     │
    │                                                                        │
    │ • Goal: predict a label for every time sample (e.g., “noise / P / S”). │
    │ • Shape convention: (batch, channels, length) — PyTorch’s default.     │
    │ • Two big parts:                                                       │
    │     Encoder (Down path): “What is present?”                            │
    │     Decoder (Up path)  : “Where exactly is it?”                        │
    │   Skip connections copy high-resolution info from encoder to decoder.  │
    └────────────────────────────────────────────────────────────────────────┘
    """
    def __init__(self, 
                 in_channels = 3,               # e.g., 3-component seismogram
                 out_channels = 3,              # e.g., P-wave, S-wave, noise
                 features = [16, 32, 64, 182]   # network width; doubles every step by default
                 ):
        super().__init__()

        # ==============================
        # 1️⃣ Downsampling Path (ENCODER)
        # ==============================
        # Build the ENCODER (“Downs”) — series of ConvBlock + MaxPool  
        #  • Each ConvBlock learns richer features                     
        #  • MaxPool (done later in `forward`) halves time resolution
        #      doubling the “receptive field” (context window).
        # ---------------------------------------------------------------
        self.downs = nn.ModuleList()
        for feat in features:
            self.downs.append(ConvBlock(in_channels, feat)) # using the ConvBlock function we defined earlier
            in_channels = feat          # update in_channels for the next block where we are incrasing the features

        # ============================================
        # 2️⃣ Bottleneck (connects ENCODER & DECODER)
        # ============================================
        # bottleneck refers to the deepest layer in the U-Net, connects encoder and decoder
        #    • Sees the shortest signal (most compressed) but richest channels.
        #    • Doubles channels one last time.
        # --------------------------------------------------------------------------------------
        self.bottleneck = ConvBlock(features[-1], features[-1]*2)

        # ==============================
        # 3️⃣ Upsampling path (DECODER)
        # ==============================
        # Build the DECODER (“Ups”) — mirror of encoder
        #    For every level we create two layers:
        #       a) ConvTranspose1d for learnable upsampling (×2 length)
        #       b) ConvBlock to fuse the upsampled data with a skip connection
        # ------------------------------------------------------------------------
        self.ups   = nn.ModuleList()
        for feat in reversed(features):         # traverse 128→64→32→16
            # a) Up-convolution (upsample via transposed convolution): halves channels, doubles length
            self.ups.append(nn.ConvTranspose1d(feat*2, feat, kernel_size = 2, stride = 2))
            # b) ConvBlock: input has 2×feat channels (feat from up + feat skip)            
            self.ups.append(ConvBlock(feat*2, feat))

        # ===========================
        # Final output convolution
        # ===========================
        #    • Acts like a fully-connected layer applied at each time step.
        # ----------------------------------------------------------------------
        self.final_conv = nn.Conv1d(features[0], out_channels, kernel_size=1)

    # ════════════════════════════════════════════════════════════════════════
    # Forward pass: Encoder ➜ Bottleneck ➜ Decoder ➜ Classifier
    # ════════════════════════════════════════════════════════════════════════
    def forward(self, x):
        skip_stack = []         # will collect encoder outputs for skip connections

        # ---------------- Encoder ----------------
        for down in self.downs:
            x = down(x)                             # ConvBlock (keeps length)
            skip_stack.append(x)                    # save high-res features
            x = F.maxpool1d(x, kernel_size = 2)     # ↓2: halve length, double context
        
        # --------------- Bottleneck ---------------
        x = self.bottleneck(x)

        # Reverse list so the first pop corresponds to the last encoder block
        skip_stack = skip_stack[::-1]

        # ---------------- Decoder ----------------
        # Iterate pair-wise: (upconv, convblock), (upconv, convblock), ...
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)                # (a) learnable upsampling
            skip_conn = skip_stack[idx//2]      # (b) matching skip feature map
            
            # If the lengths differ by 1 (can happen with odd numbers),
            # right-pad the upsampled tensor so they match.
            if x.shape[-1] != skip_conn.shape[-1]:
                x = F.pad(x, (0, skip_conn.shape[-1] - x.shape[-1]))

            # Concatenate along channel dimension: [skip | upsampled]            
            x = torch.cat((skip_conn, x), dim = 1)
            x = self.ups[idx+1](x)
        
        x = self.final_conv(x)          # raw scores per class
        return F.softmax(x, dim = 1)    # convert to probabilities

##### **Reading tips for absolute beginners**

1. Tensors vs. Arrays – Think of `torch.Tensor` as an upgraded NumPy array that also knows how to compute gradients for learning.

2. Channels vs. Length – Image U-Nets slide a 2-D window over height×width; here we slide a 1-D window over time. `channels` are like different “color layers” of the signal (e.g., E-N-Z components).

3. Why halve length then double it again? ans: Shrinking lets the network “see” a bigger context cheaply. Growing back recovers the original resolution so the final answer aligns sample-wise with the input.

4. Skip connections – Without them, the decoder would only see blurred, low-resolution data and might mis-locate events. Skips inject sharp details from early layers.

5. ConvTranspose1d – Often called “deconvolution” or “up-conv”. It is learned upsampling: instead of interpolating, it finds the best way to spread compressed information back over time.

now, we are going to bring over the analyze_earthquake function here. But this time we are only going to use the S3 bucket over the cloud, since we have already decleared cloud supremacy in the previous notebooks

In [4]:
# ----------------------------------------------
# Import Required Libraries
# ----------------------------------------------
import os
import dask.delayed
import numpy as np
from obspy import UTCDateTime
from obspy import read
from obspy.clients.fdsn.client import Client
from obspy.core.inventory.inventory import Inventory
from scipy import signal
import seisbench.models as sbm  # Import PhaseNet model from SeisBench
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError
import dask
from io import BytesIO
from torch.utils.data import Dataset, DataLoader

# ----------------------------------------------
# Configure S3 Client for Public NCEDC Access
# ----------------------------------------------
s3 = boto3.client('s3', config = Config(signature_version = UNSIGNED), region_name='us-west-2')
BUCKET_NAME = 'ncedc-pds'

# -------------------------------------------------------------------------
# 📌 Utility Function to check if data is available in repository
# -------------------------------------------------------------------------
def station_data_exists(station, eq_time, pre_time, post_time, client: Client, network: str, location: str, channel: str,
                        s3_client, bucket_name: str) -> bool:

    # S3 check — build the same key you use in your main function
    day0 = eq_time.replace(hour=0, minute=0, second=0, microsecond=0)
    jul   = day0.julday
    fname = f"{station.code}.{network}.{channel}..D.{day0.year}.{jul:03d}"
    key   = f"continuous_waveforms/{network}/{day0.year}/" \
            f"{day0.year}.{jul:03d}/{fname}"
    try:
        s3_client.head_object(Bucket=bucket_name, Key=key)
    except ClientError:
        return False

    return True

# --------------------------------------------------------------
# 📌 Utility Function to Filter Inventory Based on Data Availability
# --------------------------------------------------------------
def filter_inventory(inventory: Inventory, eq_time, pre_time, post_time, 
                     client: Client, network: str, location: str, channel: str,
                     s3_client, bucket_name: str) -> Inventory:
    
    # iterate all networks
    kept_networks = []
    for net in inventory.networks:
        kept_stns = []
        for st in net.stations:
            if station_data_exists(st, eq_time, pre_time, post_time, client, network, location, channel,
                                   s3_client, bucket_name):
                kept_stns.append(st)
        if kept_stns:
            net.stations = kept_stns
            kept_networks.append(net)

    inventory.networks = kept_networks
    return inventory

# ----------------------------------------------
# Load Pretrained Phase Picker (PhaseNet)
# ----------------------------------------------
picker = sbm.PhaseNet.from_pretrained("original")

# --------------------------------------------------
# --------------------------------------------------
# ✂️ SEPERATE FUNCTION TO PROCESS STATIONS IN PARALLEL
# --------------------------------------------------
# --------------------------------------------------
def process_stations(station, start_time, pre_time, post_time, eq_time,
                     network, channel, inventory):
    station_code = station.code
    # -----------------------------------------------------------
    # Step 2️⃣: # Download waveforms & apply instrument correction
    # -----------------------------------------------------------        
    file_name = f'{station_code}.{network}.{channel}..D.{start_time.year}.{start_time.julday:03d}'
    KEY = f"continuous_waveforms/{network}/{start_time.year}/{start_time.year}.{start_time.julday:03d}/{file_name}"

    # stream the object from S3 and wrap in a BytesIO
    resp = s3.get_object(Bucket=BUCKET_NAME, Key=KEY)
    data_stream = resp['Body']              # this is a file-like StreamingBody
    buff = BytesIO(data_stream.read())      # read all bytes into an in-memory buffer
    buff.seek(0)                            # rewind to the front
    
    # now read directly from that buffer
    st_stream = read(buff, format='MSEED')
    st_stream.trim(starttime=eq_time - pre_time, endtime=eq_time + post_time)

    print(f"- Downloaded {len(st_stream)} traces for station {station_code}.")

    # Assuming single trace per station
    tr = st_stream[0]

    # Remove the instrument response to convert counts to ground displacement (in meters)
    tr.remove_response(inventory=inventory, output="DISP")

    # ----------------------------------------------
    # Step 3️⃣: Pick P-wave Arrivals & Slice Waveform
    # ----------------------------------------------
    picks = picker.classify(st_stream, batch_size=256, P_threshold=0.075, S_threshold=0.1).picks
    if not picks:
        raise Exception(f"- No picks found for station {station_code}.")
    
    # Use first P arrival time for plotting
    p_time = picks[0].peak_time

    manifest.append({
        'bucket': BUCKET_NAME,
        'key':    KEY,
        'pick_time': p_time, 'pre_time':  pre_time, 'post_time': post_time, 'eq_time':   eq_time
    })

# ----------------------------------------------
# 🧰 Main Function: Seismogram Analysis Workflow
# ----------------------------------------------
manifest = []
def analyze_earthquake_manifest(eq_time, eq_lon, eq_lat, radius_km,
                       client_name='NCEDC', network='NC', location='*', channel='HNE',
                       pre_time=3, post_time=120, output_dir='plots/dask', use_dask=True):

    # Ensure eq_time is UTCDateTime
    if not isinstance(eq_time, UTCDateTime):
        eq_time = UTCDateTime(eq_time)

    # Define waveform time window
    start_time = eq_time.replace(hour=0, minute=0, second=0, microsecond=0)
    end_time = eq_time.replace(hour=23, minute=59, second=59, microsecond=999999)

    # Create output directory if it doesn't exist
    os.makedirs(f"{output_dir}", exist_ok=True)

    # ----------------------------------------------
    # Step 1️⃣: Retrieve Station Metadata
    # ----------------------------------------------
    client = Client(client_name)
    print("Making inventory of stations ...")
    inventory = client.get_stations(network=network, latitude=eq_lat, longitude=eq_lon,
                                    starttime=start_time, endtime=end_time, maxradius=radius_km/111.2, # Convert km to degrees
                                    location=location, channel=channel, level="response")
    
    print("Filtering the inventory ...")
    inventory = filter_inventory(inventory, eq_time, pre_time, post_time, client, network, location, channel, s3, BUCKET_NAME)

    stations = inventory[0].stations
    print(f"Found {len(stations)} stations within {radius_km} km of ({eq_lat}, {eq_lon}).")

    # ----------------------------------------------
    # Step 2️⃣: Loop Over Each Station using DASK
    # ----------------------------------------------

    for st in stations:
            process_stations(st, start_time, pre_time, post_time, eq_time,
                     network, channel, inventory)
    
    return manifest

In [ ]:
# ── 3) Put it all together ─────────────────────────────────────────────────────
manifest = analyze_earthquake_manifest(
    '2022-12-20T10:34:24', -124.588, 40.369, 200,
    client_name='NCEDC',
    network='NC', location='*', channel='HNE',
    pre_time=3, post_time=120
)

Making inventory of stations ...
Filtering the inventory ...
Found 8 stations within 200 km of (40.369, -124.588).
- Downloaded 1 traces for station KCO.
- Downloaded 1 traces for station KCT.
- Downloaded 1 traces for station KHBB.
- Downloaded 1 traces for station KHMB.
- Downloaded 1 traces for station KMPB.
- Downloaded 1 traces for station KMR.
- Downloaded 1 traces for station KRMB.
- Downloaded 1 traces for station KSXB.


In [13]:
# Custom Dataset class for seismic data
class SeismicDataset(Dataset):
    def __init__(self, manifest, window_size=5):
        """
        Args:
            manifest: List of processed station data
            window_size: Size of window around P-pick for positive labels (in seconds)
        """
        self.data = []
        self.labels = []
        self.window_size = window_size
        
        for item in manifest:
            waveform = item['waveform']
            sampling_rate = item['sampling_rate']
            pick_time = item['pick_time']
            eq_time = item['eq_time']
            pre_time = item['pre_time']
            
            # Calculate pick sample index
            pick_offset = (pick_time - (eq_time - pre_time))  # seconds from start
            pick_sample = int(pick_offset * sampling_rate)
            
            # Create labels (0 = background, 1 = P-wave)
            label = np.zeros(len(waveform))
            window_samples = int(window_size * sampling_rate / 2)  # ±window_size/2 seconds
            
            start_idx = max(0, pick_sample - window_samples)
            end_idx = min(len(waveform), pick_sample + window_samples)
            label[start_idx:end_idx] = 1
            
            self.data.append(waveform)
            self.labels.append(label)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        waveform = torch.FloatTensor(self.data[idx]).unsqueeze(0)  # Add channel dimension
        label = torch.LongTensor(self.labels[idx])
        return waveform, label

In [6]:
# ── 2) The S3‐streaming Dataset ──────────────────────────────────────────────────
class S3PhasePickDataset(Dataset):
    def __init__(self, manifest, inventory, network, location, channel):
        self.manifest = manifest
        self.s3        = boto3.client('s3', config=Config(signature_version=UNSIGNED), region_name='us-west-2')
        self.inventory = inventory             # full Inventory (with responses)
        self.network   = network
        self.location  = location
        self.channel   = channel

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, i):
        ex = self.manifest[i]
        # 1) stream waveform
        resp = self.s3.get_object(Bucket=ex['bucket'], Key=ex['key'])
        buff = BytesIO(resp['Body'].read())
        st   = read(buff, format='MSEED')
        tr   = st[0]

        # 2) re‐trim exactly around pick window
        tr.trim(starttime=ex['pick_time'] - ex['pre_time'],
                endtime=ex['pick_time'] + ex['post_time'])
        
        # 3) instrument correction
        tr.remove_response(inventory=self.inventory, output="DISP")

        data = tr.data.astype('float32')
        sr   = tr.stats.sampling_rate
        # 4) label sequence
        pick_idx = int((ex['pick_time'] - tr.stats.starttime) * sr)
        labels   = make_label_sequence(len(data), pick_idx)

        # 5) to torch
        x = torch.from_numpy(data).unsqueeze(0)   # (1, L)
        return x, labels  

In [14]:
# Training function
def train_model(model, train_loader, val_loader, num_epochs=50, learning_rate=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_batches = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_batches += 1
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_batches = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                val_loss += criterion(output, target).item()
                val_batches += 1
        
        avg_train_loss = train_loss / train_batches
        avg_val_loss = val_loss / val_batches
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
    
    return train_losses, val_losses

In [15]:
# Evaluation function
def evaluate_picks(model, val_dataset, threshold=0.5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    pick_errors = []
    
    with torch.no_grad():
        for i in range(len(val_dataset)):
            waveform, true_label = val_dataset[i]
            waveform = waveform.unsqueeze(0).to(device)  # Add batch dimension
            
            output = model(waveform)
            prob = output[0, 1, :].cpu().numpy()  # P-wave probability
            
            # Find predicted pick (maximum probability)
            pred_pick_sample = np.argmax(prob)
            
            # Find true pick (center of labeled window)
            true_pick_samples = np.where(true_label == 1)[0]
            if len(true_pick_samples) > 0:
                true_pick_sample = np.mean(true_pick_samples)
                
                # Calculate error in samples
                error_samples = abs(pred_pick_sample - true_pick_sample)
                pick_errors.append(error_samples)
    
    return np.array(pick_errors)

In [17]:
# Main training pipeline
def main():
    # Example earthquakes (you can add more)
    earthquakes = [
        {"time": '2022-12-20T10:34:24', "lat": 40.369, "lon": -124.588, "radius": 500}, # Frendale
        {"time": "2010-01-10T00:27:39", "lat": 40.652, "lon": -124.693, "radius": 500}, # Eureka
        {"time": "2014-08-24T10:20:44", "lat": 38.215, "lon": -122.312, "radius": 500}, # South Napa
    ]
    
    # Collect data from multiple earthquakes
    all_manifest = []
    for eq in earthquakes:
        print(f"\nProcessing earthquake: {eq['time']}")
        manifest = analyze_earthquake_manifest(
            eq_time=eq['time'],
            eq_lat=eq['lat'],
            eq_lon=eq['lon'],
            radius_km=eq['radius']
        )
        all_manifest.extend(manifest)
        print(f"Collected {len(manifest)} stations for this earthquake")
    
    print(f"\nTotal stations collected: {len(all_manifest)}")
    
    if len(all_manifest) == 0:
        print("No data collected. Exiting.")
        return
    
    # Split data into training and validation (80/20)
    train_manifest, val_manifest = train_test_split(all_manifest, test_size=0.2, random_state=42)
    
    print(f"Training samples: {len(train_manifest)}")
    print(f"Validation samples: {len(val_manifest)}")
    
    # Create datasets
    train_dataset = SeismicDataset(train_manifest, window_size=5)
    val_dataset = SeismicDataset(val_manifest, window_size=5)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
    
    # Initialize model
    model = UNet1D(in_channels=1, out_channels=2)
    
    # Train model
    print("\nStarting training...")
    train_losses, val_losses = train_model(model, train_loader, val_loader, num_epochs=20)
    
    # Evaluate model
    print("\nEvaluating model...")
    pick_errors = evaluate_picks(model, val_dataset)
    
    # Assuming 100 Hz sampling rate for error calculation
    sampling_rate = 100  # Hz
    pick_errors_seconds = pick_errors / sampling_rate
    
    print(f"\nPick Time Accuracy Results:")
    print(f"Mean absolute error: {np.mean(pick_errors_seconds):.3f} seconds")
    print(f"Standard deviation: {np.std(pick_errors_seconds):.3f} seconds")
    print(f"Median absolute error: {np.median(pick_errors_seconds):.3f} seconds")
    
    # Plot training curves
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.hist(pick_errors_seconds, bins=30, alpha=0.7)
    plt.xlabel('Pick Time Error (seconds)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Pick Time Errors')
    
    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Save model
    torch.save(model.state_dict(), 'unet_phase_picker.pth')
    print("Model saved as 'unet_phase_picker.pth'")

if __name__ == "__main__":
    main()


Processing earthquake: 2022-12-20T10:34:24
Making inventory of stations ...
Filtering the inventory ...
Found 77 stations within 500 km of (40.369, -124.588).
- Downloaded 1 traces for station AFD.
- Downloaded 1 traces for station BSR.
- Downloaded 1 traces for station CADB.
- Downloaded 1 traces for station CAG.
- Downloaded 1 traces for station CBP.
- Downloaded 1 traces for station CBR.
- Downloaded 1 traces for station CCOB.
- Downloaded 1 traces for station CGP.
- Downloaded 0 traces for station CHR.


IndexError: list index out of range

In [5]:
Inventory

obspy.core.inventory.inventory.Inventory

In [7]:

# reuse the same inventory you passed in above so that remove_response works:
dataset = S3PhasePickDataset(manifest, Inventory, network='NC', location='*', channel='HNE')
loader  = DataLoader(dataset, batch_size=8, shuffle=True)

# then your training loop is identical:
model     = UNet1D(in_channels=1, out_channels=2, features=[16,32,64,128]).to('cpu')
criterion = torch.nn.CrossEntropyLoss()
opt       = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1,21):
    model.train()
    for x,y in loader:
        x,y = x.to('cpu'), y.to('cpu')
        opt.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        opt.step()
    # … validation …

TypeError: 'type' object is not iterable

In [12]:
loader